# ERP Stok Kartı Tekilleştirme - İlk Veri Analizi

Bu notebook, ERP sisteminden alınan anonim stok kartı verisi üzerinde ilk veri inceleme, temizleme ve benzerlik analizi adımlarını içerir.

In [55]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Veri yolu - GitHub reposunda dosyayı data klasörüne koy
DATA_PATH = 'data.xlsx'

df = pd.read_excel(DATA_PATH)
df.head()

,STOK KODU,STOK ADI,YALIN HALİ,KOD UZUNLUĞU
0,UCY 10H125114,İNTERCOL HORTUM,UCY10H125114,13
1,BSCH 1 987 947 948,V KAYIS KANALLI 6PK1460,BSCH1987947948,19
2,UCY 10H125113,INTERCOOLER HORTUM,UCY10H125113,13
3,UCY 10H12526,INTERCOOLER HORTUM,UCY10H12526,12
4,UCY 10H12528,INTERCOOLER HORTUM,UCY10H12528,12


In [56]:
import sys
print(sys.executable)

import openpyxl
print(openpyxl.__version__)

c:\Users\Dost\AppData\Local\Programs\Python\Python313\python.exe
3.1.5


In [57]:
# Veri seti genel bilgileri
print('Satır / Sütun:', df.shape)
print('\nKolonlar:')
print(df.columns.tolist())

print('\nEksik veri sayıları:')
print(df.isnull().sum())

Satır / Sütun: (3558, 4)

Kolonlar:
['STOK KODU', 'STOK ADI', 'YALIN HALİ', 'KOD UZUNLUĞU']

Eksik veri sayıları:
STOK KODU       0
STOK ADI        1
YALIN HALİ      0
KOD UZUNLUĞU    0
dtype: int64


In [58]:
# Temel temizlik
df['STOK ADI'] = df['STOK ADI'].astype(str).str.strip().str.upper()
df['YALIN HALİ'] = df['YALIN HALİ'].astype(str).str.strip().str.upper()
df['STOK KODU'] = df['STOK KODU'].astype(str).str.strip().str.upper()

df.head()

,STOK KODU,STOK ADI,YALIN HALİ,KOD UZUNLUĞU
0,UCY 10H125114,İNTERCOL HORTUM,UCY10H125114,13
1,BSCH 1 987 947 948,V KAYIS KANALLI 6PK1460,BSCH1987947948,19
2,UCY 10H125113,INTERCOOLER HORTUM,UCY10H125113,13
3,UCY 10H12526,INTERCOOLER HORTUM,UCY10H12526,12
4,UCY 10H12528,INTERCOOLER HORTUM,UCY10H12528,12


In [59]:
# Aynı stok adına sahip kayıtları bulma
duplicate_names = df[df.duplicated(subset=['STOK ADI'], keep=False)].sort_values('STOK ADI')
duplicate_names[['STOK KODU', 'STOK ADI', 'YALIN HALİ']].head(20)

,STOK KODU,STOK ADI,YALIN HALİ
291,BK21 V200K22 AD N,(RHD) PANEL - GOVDE YAN - ARKA,BK21V200K22ADN
2397,BK21 V200K22 AD N,(RHD) PANEL - GOVDE YAN - ARKA,BK21V200K22ADN
2037,74780 N,12 DIS MARS DISLISI,74780N
2327,74780 N,12 DIS MARS DISLISI,74780N
2213,AMP FORD 007 N,12 OLT H7 AMPUL,AMPFORD007N
279,AMP FORD 007 N,12 OLT H7 AMPUL,AMPFORD007N
2321,64150 CB N,12 V. H1 BEYAZ AMPUL,64150CBN
1739,64150 CB N,12 V. H1 BEYAZ AMPUL,64150CBN
2143,64211 LTS N,12 V. H11 AMPUL,64211LTSN
1737,64211 LTS N,12 V. H11 AMPUL,64211LTSN


In [60]:
# Stok adı bazlı TF-IDF + Cosine Similarity
# Amaç: yazımı benzer olan stok kartlarını skorlayarak tespit etmek

texts = df['STOK ADI'].fillna('').astype(str)

vectorizer = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5))
tfidf_matrix = vectorizer.fit_transform(texts)

similarity_matrix = cosine_similarity(tfidf_matrix)

similarity_matrix.shape

(3558, 3558)

In [61]:
# Belirli bir stok kaydı için en benzer kayıtları listeleme

def get_similar_items(index, top_n=10):
    scores = list(enumerate(similarity_matrix[index]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)
    scores = [s for s in scores if s[0] != index]
    top_scores = scores[:top_n]
    
    results = []
    for item_index, score in top_scores:
        results.append({
            'ana_stok_kodu': df.iloc[index]['STOK KODU'],
            'ana_stok_adi': df.iloc[index]['STOK ADI'],
            'benzer_stok_kodu': df.iloc[item_index]['STOK KODU'],
            'benzer_stok_adi': df.iloc[item_index]['STOK ADI'],
            'benzerlik_skoru': round(score, 4)
        })
    return pd.DataFrame(results)

get_similar_items(0, top_n=10)

,ana_stok_kodu,ana_stok_adi,benzer_stok_kodu,benzer_stok_adi,benzerlik_skoru
0,UCY 10H125114,İNTERCOL HORTUM,UCY 10 H 125114,İNTERCOOL HORTUM Y.M,0.6905
1,UCY 10H125114,İNTERCOL HORTUM,BK21 8C012 BB N,HORTUM,0.5184
2,UCY 10H125114,İNTERCOL HORTUM,F1DQ 9S468 AC N,HORTUM,0.5184
3,UCY 10H125114,İNTERCOL HORTUM,BK21 8C012 BB N,HORTUM,0.5184
4,UCY 10H125114,İNTERCOL HORTUM,F1DQ 9S468 AC N,HORTUM,0.5184
5,UCY 10H125114,İNTERCOL HORTUM,UCY 10H125113,INTERCOOLER HORTUM,0.4670
6,UCY 10H125114,İNTERCOL HORTUM,UCY 10H12526,INTERCOOLER HORTUM,0.4670
7,UCY 10H125114,İNTERCOL HORTUM,UCY 10H12528,INTERCOOLER HORTUM,0.4670
8,UCY 10H125114,İNTERCOL HORTUM,UCY 10 H 125113,HORTUM INTERCOOL Y.M,0.4642
9,UCY 10H125114,İNTERCOL HORTUM,UCY 10 H125114,HORTUM INTERCOOL Y.M,0.4642


In [62]:
# Tüm veri için belirli eşik üzerindeki benzer kayıt çiftlerini bulma

threshold = 0.95
pairs = []

for i in range(len(df)):
    for j in range(i + 1, len(df)):
        score = similarity_matrix[i, j]
        if score >= threshold:
            pairs.append({
                'stok_kodu_1': df.iloc[i]['STOK KODU'],
                'stok_adi_1': df.iloc[i]['STOK ADI'],
                'stok_kodu_2': df.iloc[j]['STOK KODU'],
                'stok_adi_2': df.iloc[j]['STOK ADI'],
                'benzerlik_skoru': round(score, 4)
            })

similar_pairs = pd.DataFrame(pairs)
similar_pairs.head(20)

,stok_kodu_1,stok_adi_1,stok_kodu_2,stok_adi_2,benzerlik_skoru
0,UCY 10H125113,INTERCOOLER HORTUM,UCY 10H12526,INTERCOOLER HORTUM,1.0
1,UCY 10H125113,INTERCOOLER HORTUM,UCY 10H12528,INTERCOOLER HORTUM,1.0
2,UCY 10H12526,INTERCOOLER HORTUM,UCY 10H12528,INTERCOOLER HORTUM,1.0
3,W712889 S300 N,KLIPS,W702806 S442 N,KLIPS,1.0
4,W712889 S300 N,KLIPS,W715271 S300 N,KLIPS,1.0
5,W712889 S300 N,KLIPS,W715667 S439 N,KLIPS,1.0
6,W712889 S300 N,KLIPS,W716740 S439 N,KLIPS,1.0
7,W712889 S300 N,KLIPS,W527005 S437M N,KLIPS,1.0
8,W712889 S300 N,KLIPS,W716740 S439 N,KLIPS,1.0
9,W712889 S300 N,KLIPS,W715271 S300 N,KLIPS,1.0


In [63]:
# Sonuçları CSV olarak kaydetme
similar_pairs.to_csv('benzer_stok_kartlari.csv', index=False, encoding='utf-8-sig')

print('Benzer stok kartı çifti sayısı:', len(similar_pairs))

Benzer stok kartı çifti sayısı: 10815


In [64]:
# Modelin bulduğu ilk 50 benzer çifti manuel kontrol için alıyoruz
manual_check = similar_pairs.head(50).copy()

# Gerçek durum kolonunu ekliyoruz
# Buraya elle 1 veya 0 yazılacak:
# 1 = gerçekten aynı / duplicate
# 0 = aynı değil
manual_check["gercek_durum"] = "1"

manual_check.to_csv("manuel_dogrulama_seti.csv", index=False, encoding="utf-8-sig")

manual_check.head()

,stok_kodu_1,stok_adi_1,stok_kodu_2,stok_adi_2,benzerlik_skoru,gercek_durum
0,UCY 10H125113,INTERCOOLER HORTUM,UCY 10H12526,INTERCOOLER HORTUM,1.0,1
1,UCY 10H125113,INTERCOOLER HORTUM,UCY 10H12528,INTERCOOLER HORTUM,1.0,1
2,UCY 10H12526,INTERCOOLER HORTUM,UCY 10H12528,INTERCOOLER HORTUM,1.0,1
3,W712889 S300 N,KLIPS,W702806 S442 N,KLIPS,1.0,1
4,W712889 S300 N,KLIPS,W715271 S300 N,KLIPS,1.0,1


In [65]:
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
import pandas as pd

eval_df = pd.read_csv("manuel_dogrulama_seti.csv")

# boş etiketleri temizle
eval_df = eval_df.dropna(subset=["gercek_durum"])

# sadece 0 ve 1 olanları al
eval_df = eval_df[eval_df["gercek_durum"].isin([0, 1, "0", "1"])]

eval_df["model_tahmini"] = 1
eval_df["gercek_durum"] = eval_df["gercek_durum"].astype(int)

precision = precision_score(eval_df["gercek_durum"], eval_df["model_tahmini"])
recall = recall_score(eval_df["gercek_durum"], eval_df["model_tahmini"])
f1 = f1_score(eval_df["gercek_durum"], eval_df["model_tahmini"])
accuracy = accuracy_score(eval_df["gercek_durum"], eval_df["model_tahmini"])

print("Değerlendirilen kayıt sayısı:", len(eval_df))
print("Precision:", round(precision, 3))
print("Recall:", round(recall, 3))
print("F1-score:", round(f1, 3))
print("Accuracy:", round(accuracy, 3))

Değerlendirilen kayıt sayısı: 50
Precision: 1.0
Recall: 1.0
F1-score: 1.0
Accuracy: 1.0


In [66]:
eval_df["gercek_durum"].value_counts(dropna=False)

gercek_durum
1    50
Name: count, dtype: int64